# FaceForensics++ Dataset(c23)

## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading the **FaceForensics++** dataset. We have chosen the **c23** version, which corresponds to a moderate H.264 video compression (Constant Rate Factor 23). 

To streamline the acquisition process, we rely on the `kagglehub` library. This tool allows us to fetch the dataset directly from Kaggle and caches it locally.

In [1]:
import kagglehub
import os

print("Starting the download of the FaceForensics++ (c23) dataset...")
print("Note: This might take a while the first time...")

# Download the dataset directly using kagglehub
dataset_path = kagglehub.dataset_download("xdxd003/ff-c23")
print("Download completed or loaded from chache")
print("Dataset path:", dataset_path)

#Contents of the downloaded folder to see what we have
folder_items = os.listdir(dataset_path)
print("Contents inside", dataset_path, ": ")
print('\n'.join([f"- {item}" for item in folder_items]))

/Users/luciapola/Desktop/internship-deepfake-forensic/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/luciapola/Desktop/internship-deepfake-forensic/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting the download of the FaceForensics++ (c23) dataset...
Note: This might take a while the first time...
Download completed or loaded from chache
Dataset path: /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1
Contents inside /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1 : 
- FaceForensics++_C23


In [14]:
import glob

base_path = os.path.join(dataset_path, "FaceForensics++_C23")
folders = os.listdir(base_path)
print("Contents inside", base_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))


Contents inside /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23 : 
- DeepFakeDetection
- Deepfakes
- original
- FaceShifter
- FaceSwap
- csv
- NeuralTextures
- Face2Face


In [3]:
for folder in folders:
    folder_path = os.path.join(base_path, folder)

    try:
        files = [file for file in os.listdir(folder_path)]

        if files:
            print(f"{folder}: {files[0]}")
        else:
            print(f"{folder}: Zero .mp4 files founded")

    except FileNotFoundError:
        print(f"{folder}: folder not founded")

DeepFakeDetection: 02_15__walking_and_outside_surprised__MZWH8ATN.mp4
Deepfakes: 475_265.mp4
original: 578.mp4
FaceShifter: 475_265.mp4
FaceSwap: 475_265.mp4
csv: NeuralTextures.csv
NeuralTextures: 475_265.mp4
Face2Face: 475_265.mp4


## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all videos from the dataset.

Columns:
- `video`: filename
- `label`: 0 = real, 1 = fake
- `method`: manipulation type or "original"
- `target`: ID of the person being manipulated (for fakes) or identity of original
- `source`: ID of the source video for fakes, None for real videos
- `path`: full path to the video file

In [4]:
import pandas as pd

data = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    if not os.path.exists(folder_path):
        continue

    if folder == "original":
        for video in os.listdir(folder_path):
            data.append({
                "video": video,
                "label": 0,
                "method": "original",
                "target": int(video.replace(".mp4", "")),
                "source": None,
                "sequence": None,
                "exp_id": None,
                "path": os.path.join(folder_path, video)
            })

    elif folder == "DeepFakeDetection":
        for video in os.listdir(folder_path):
            if not video.endswith(".mp4"): continue

            name = video.replace(".mp4", "")
            parts = name.split("__")

            if len(parts) >= 3:
                actors = parts[0]
                sequence = parts[1]
                exp_id = parts[2]

                actors_parts = actors.split("_")
                if len(actors_parts) >= 2:
                    target = actors_parts[0]
                    source = actors_parts[1]
            else:
                continue

            data.append({
                "video": video,
                "label": 1,
                "method": folder,
                "target": target,
                "source": source,
                "sequence": sequence,
                "exp_id": exp_id,
                "path": os.path.join(folder_path, video)
            })

    else:
        for video in os.listdir(folder_path):
            if not video.endswith(".mp4"): continue

            name = video.replace(".mp4", "")
            parts = name.split("_")
            if len(parts) >= 2:
                target = parts[0]
                source = parts[1]
            else:
                continue

            data.append({
                "video": video,
                "label": 1,
                "method": folder,
                "target": target,
                "source": source,
                "sequence": None,
                "exp_id": None,
                "path": os.path.join(folder_path, video)
            })
    
videos = pd.DataFrame(data)

pd.set_option('display.max_colwidth', None)
display(videos.sample(15))

,video,label,method,target,source,sequence,exp_id,path
4370,808_829.mp4,1,FaceSwap,808,829,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceSwap/808_829.mp4
4366,937_888.mp4,1,FaceSwap,937,888,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceSwap/937_888.mp4
3329,997_040.mp4,1,FaceShifter,997,040,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/997_040.mp4
4221,951_947.mp4,1,FaceSwap,951,947,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceSwap/951_947.mp4
5824,031_163.mp4,1,NeuralTextures,031,163,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/031_163.mp4
17,14_26__kitchen_still__ILLP29ZH.mp4,1,DeepFakeDetection,14,26,kitchen_still,ILLP29ZH,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/14_26__kitchen_still__ILLP29ZH.mp4
850,07_21__outside_talking_still_laughing__K7KXUHMU.mp4,1,DeepFakeDetection,07,21,outside_talking_still_laughing,K7KXUHMU,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/07_21__outside_talking_still_laughing__K7KXUHMU.mp4
720,19_23__walking_outside_cafe_disgusted__WHQ1229T.mp4,1,DeepFakeDetection,19,23,walking_outside_cafe_disgusted,WHQ1229T,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/19_23__walking_outside_cafe_disgusted__WHQ1229T.mp4
5501,482_465.mp4,1,NeuralTextures,482,465,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/482_465.mp4
768,06_18__walking_down_street_outside_angry__MVXBDKGB.mp4,1,DeepFakeDetection,06,18,walking_down_street_outside_angry,MVXBDKGB,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/06_18__walking_down_street_outside_angry__MVXBDKGB.mp4


### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly:

- Total number of videos
- Distribution of real vs fake videos
- Distribution of manipulation methods
- Number of unique target and source IDs

In [15]:
print("Total videos:", len(videos))
print("\nLabel distribution (fake/real):")
print(videos["label"].value_counts())
print("\nMethod distribution:")
print(videos["method"].value_counts())
print("\nTarget unique number (identity):", videos["target"].nunique())
print("\nSource unique number:", videos["source"].nunique())

Total videos: 7000

Label distribution (fake/real):
label
1    6000
0    1000
Name: count, dtype: int64

Method distribution:
method
DeepFakeDetection    1000
Deepfakes            1000
original             1000
FaceShifter          1000
FaceSwap             1000
NeuralTextures       1000
Face2Face            1000
Name: count, dtype: int64

Target unique number (identity): 2028

Source unique number: 1028


### 2.2  - Distribution of Fake Videos per Target

Analyzes how many fake videos exist per target identity.

Purpose:
- Identify if some identities dominate the fake samples
- Helps to plan balanced train/test splits

In [6]:
fake_df = videos[videos["label"] == 1]
per_target = fake_df.groupby("target").size()
print("\n--- FAKE PER TARGET STATISTICS ---")
print(per_target.describe())


--- FAKE PER TARGET STATISTICS ---
count    1028.000000
mean        5.836576
std         5.854495
min         5.000000
25%         5.000000
50%         5.000000
75%         5.000000
max        76.000000
dtype: float64


### 2.3 - Distribution Of Methods per Target (Check Variability):

Creates a pivot table showing how many videos of each manipulation method exist per target.

Purpose:
- Verify that each identity has a representative set of manipulation methods
- Detect identities with too few or missing manipulation types

In [16]:
pivot = pd.pivot_table(
    videos,
    index="target",
    columns="method",
    values="video",
    aggfunc="count",
    fill_value=0
)

print("\n--- METHOD DISTRIBUTION PER TARGET STATISTICS ---")
print(pivot.sample(5))


--- METHOD DISTRIBUTION PER TARGET STATISTICS ---
method  DeepFakeDetection  Deepfakes  Face2Face  FaceShifter  FaceSwap  \
target                                                                   
853                     0          1          1            1         1   
498                     0          1          1            1         1   
690                     0          0          0            0         0   
602                     0          0          0            0         0   
184                     0          0          0            0         0   

method  NeuralTextures  original  
target                            
853                  1         0  
498                  1         0  
690                  0         1  
602                  0         1  
184                  0         1  


### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset.

Purpose:
- Understand dataset imbalance

In [17]:
print("Normalize label ditribution:")
print(videos["label"].value_counts(normalize=True))

Normalize label ditribution:
label
1    0.857143
0    0.142857
Name: proportion, dtype: float64


### Dataset Observations

- **Total videos:** 7,000 in total, with 6,000 fake and 1,000 real videos. This presents a strong class imbalance (~6:1)
- **Target distribution is NOT uniform:** While the *median* is 5 fake videos per target, the dataset is heavily skewed by the `DeepFakeDetection` (DFD) subset. Because DFD relies on a small pool of professional actors filmed in multiple scenarios, some target identities appear in up to **76 fake videos**.
- **Manipulation methods:** The methods are perfectly balanced. There are exactly 1,000 videos for each of the 6 manipulation techniques (`Deepfakes`, `Face2Face`, `FaceSwap`, `NeuralTextures`, `FaceShifter`, `DeepFakeDetection`), plus the 1,000 `original` unmanipulated videos.
- **Target and source identities:** There are **2,028** unique target identities and **1,028** unique source identities. The "extra" 28 identities (beyond the base 1,000 subjects) belong to the hired actors from the Google DFD subset.

## 3 - Physical Metadata Extraction using OpenCV

In this cell, we use **OpenCV (`cv2`)** to iterate through our entire dataset of 7,000 videos and extract key metadata:
- `total_frames`
- `fps`
- `duration_sec`
- `resolution`

In [ ]:
import cv2
from tqdm import tqdm

tqdm.pandas(desc="Extracting video metadata")

videos_cv2 = videos.copy()

"""
Opens a video file momentarily just to read its metadata properties,
then closes it immediately to save memory.
"""
def get_video_metadata(video_path):
    try:
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return pd.Series([None, None, None, None])
    
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = width = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = total_frames / fps
        
        cap.release()

        return pd.Series([total_frames, fps, duration, f"{width}x{height}"])
    
    except Exception as e:
        return pd.Series([None, None, None, None])

print("Scanning all 7000 videos to extract physical metadata. This might take 1-3 minutes...")
videos[['total_frames', 'fps', 'duration_sec', 'resolution']] = videos['path'].progress_apply(get_video_metadata)

print("\nMetadata extraction complete!")

Scanning all 7000 videos to extract physical metadata. This might take 1-3 minutes...


Extracting video metadata:  22%|██▏       | 1511/7000 [00:10<00:22, 247.61it/s]

In [ ]:
print("\n--- VIDEO DURATION STATISTICS (in seconds) ---")
print(videos['duration_sec'].describe())

print("\n--- VIDEO FRAMES STATISTICS ---")
print(videos['total_frames'].describe())

display(videos.sample(20))


--- VIDEO DURATION STATISTICS (in seconds) ---
count    7000.000000
mean       19.325502
std         9.536235
min         0.208333
25%        12.833333
50%        16.500000
75%        22.320000
max        72.560000
Name: duration_sec, dtype: float64

--- VIDEO FRAMES STATISTICS ---
count    7000.000000
mean      511.797143
std       229.865219
min         5.000000
25%       346.000000
50%       444.000000
75%       594.000000
max      1814.000000
Name: total_frames, dtype: float64


,video,label,method,target,source,sequence,exp_id,path,total_frames,fps,duration_sec,resolution
2503,480.mp4,0,original,480,None,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/original/480.mp4,485,25.0,19.400000,720x720
3535,134_192.mp4,1,FaceShifter,134,192,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/134_192.mp4,346,30.0,11.533333,720x720
1599,753_789.mp4,1,Deepfakes,753,789,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Deepfakes/753_789.mp4,686,30.0,22.866667,1080x1080
5507,974_953.mp4,1,NeuralTextures,974,953,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/974_953.mp4,318,25.0,12.720000,480x480
208,04_27__walking_outside_cafe_disgusted__2CCI2ND1.mp4,1,DeepFakeDetection,04,27,walking_outside_cafe_disgusted,2CCI2ND1,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/04_27__walking_outside_cafe_disgusted__2CCI2ND1.mp4,379,24.0,15.791667,1080x1080
6839,598_178.mp4,1,Face2Face,598,178,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Face2Face/598_178.mp4,521,25.0,20.840000,480x480
3384,007_132.mp4,1,FaceShifter,007,132,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/007_132.mp4,505,25.0,20.200000,480x480
5461,661_670.mp4,1,NeuralTextures,661,670,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/661_670.mp4,481,25.0,19.240000,720x720
167,23_19__walking_down_indoor_hall_disgust__H4SUVFTL.mp4,1,DeepFakeDetection,23,19,walking_down_indoor_hall_disgust,H4SUVFTL,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/23_19__walking_down_indoor_hall_disgust__H4SUVFTL.mp4,1215,24.0,50.625000,1080x1080
3242,033_097.mp4,1,FaceShifter,033,097,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/033_097.mp4,809,25.0,32.360000,480x480


In [ ]:
import os
import pandas as pd
import glob

csv_folder = os.path.join(base_path, "csv")
csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))

print("File CSV trovati nel dataset:")
for f in csv_files:
    print(f"- {os.path.basename(f)}")

if csv_files:
    first_csv_path = csv_files[0]
    sample_csv = pd.read_csv(first_csv_path)
    
    print(f"\n--- {os.path.basename(first_csv_path)} contents ---")
    display(sample_csv.head())
    
    print("\nColonns available:", sample_csv.columns.tolist())

File CSV trovati nel dataset:
- NeuralTextures.csv
- Deepfakes.csv
- DeepFakeDetection.csv
- Mean_Data.csv
- FF++_Metadata.csv
- original.csv
- FaceSwap.csv
- FaceShifter.csv
- FF++_Metadata_Shuffled.csv
- Face2Face.csv

--- NeuralTextures.csv contents ---


,Unnamed: 0,File Path,Label,Frame Count,Width,Height,Codec,File Size(MB)
0,0,NeuralTextures/000_003.mp4,FAKE,303,640,480,h264,0.62
1,1,NeuralTextures/001_870.mp4,FAKE,460,1280,720,h264,2.71
2,2,NeuralTextures/002_006.mp4,FAKE,310,1280,720,h264,0.91
3,3,NeuralTextures/003_000.mp4,FAKE,303,640,480,h264,0.57
4,4,NeuralTextures/004_982.mp4,FAKE,309,1280,720,h264,1.19



Colonns available: ['Unnamed: 0', 'File Path', 'Label', 'Frame Count', 'Width', 'Height', 'Codec', 'File Size(MB)']


In [ ]:
import os
import pandas as pd

print("Creazione della copia videos_csv ed estrazione dati estesi dal CSV...")

videos_csv = videos.copy()

metadata_csv_path = os.path.join(base_path, "csv", "FF++_Metadata.csv")
ff_metadata = pd.read_csv(metadata_csv_path)

ff_metadata['csv_method'] = ff_metadata['File Path'].apply(lambda x: x.split('/')[0])
ff_metadata['csv_video'] = ff_metadata['File Path'].apply(lambda x: x.split('/')[-1])
ff_metadata = ff_metadata.drop_duplicates(subset=['csv_method', 'csv_video'])

ff_metadata['csv_res_string'] = ff_metadata['Width'].astype(str) + "x" + ff_metadata['Height'].astype(str)

videos_csv = videos_csv.merge(
    ff_metadata[['csv_method', 'csv_video', 'Frame Count', 'csv_res_string']], 
    left_on=['method', 'video'], 
    right_on=['csv_method', 'csv_video'], 
    how='left'
)

videos_csv = videos_csv.rename(columns={
    'Frame Count': 'csv_frames',
    'csv_res_string': 'csv_resolution'
})
videos_csv = videos_csv.drop(columns=['csv_method', 'csv_video'])

print("\nCSV extraction complete!")
print(f"Rows: {len(videos_csv)}")

Creazione della copia videos_csv ed estrazione dati estesi dal CSV...

CSV extraction complete!
Rows: 7000


In [ ]:
print("--- VERIFICATION: OpenCV vs. CSV ---")

#Comparison DataFrame
comparison_df = pd.DataFrame({
    'video': videos['video'],
    'method': videos['method'],
    'opencv_frames': videos['total_frames'],
    'csv_frames': videos_csv['csv_frames'],
    'opencv_resolution': videos['resolution'],
    'csv_resolution': videos_csv['csv_resolution']
})

mask_frames = comparison_df['opencv_frames'] != comparison_df['csv_frames']
mask_res = comparison_df['opencv_resolution'] != comparison_df['csv_resolution']

mismatched_videos = comparison_df[(mask_frames | mask_res)].dropna()

if mismatched_videos.empty:
    print("OK: Frame counts and Resolution (Width x Height) match perfectly across all 7000 videos.")
else:
    print(f"WARNING: Found discrepancies in {len(mismatched_videos)} videos.")
    display(mismatched_videos.sample(10))

--- VERIFICATION: OpenCV vs. CSV ---


,video,method,opencv_frames,csv_frames,opencv_resolution,csv_resolution
5501,482_465.mp4,NeuralTextures,329,329,480x480,640x480
785,15_12__talking_angry_couch__N0SRODQD.mp4,DeepFakeDetection,972,972,1080x1080,1920x1080
3143,717_684.mp4,FaceShifter,374,374,720x720,1280x720
4182,282_238.mp4,FaceSwap,594,594,480x480,854x480
126,09_01__talking_against_wall__O8HNNX43.mp4,DeepFakeDetection,975,975,1080x1080,1920x1080
6664,613_685.mp4,Face2Face,522,522,720x720,1280x720
665,09_20__walk_down_hall_angry__MKEWB4SM.mp4,DeepFakeDetection,309,309,1080x1080,1920x1080
1313,094_111.mp4,Deepfakes,559,559,720x720,1280x720
917,11_03__outside_talking_pan_laughing__P08VGHTA.mp4,DeepFakeDetection,688,688,1080x1080,1920x1080
6309,539_499.mp4,Face2Face,514,514,480x480,640x480


### Metadata Observations

The comparison reveals two crucial details about the dataset. First, the frame counts match perfectly between OpenCV and the CSV, confirming that the physical video files are intact and uncorrupted. However, there is a consistent discrepancy in the resolutions: the official CSV reports the original 16:9 rectangular formats (e.g., `1920x1080`), while OpenCV reveals that the actual files are 1:1 perfect squares (e.g., `1080x1080`). 